### Initialization

In [1]:
import os
import re
from pathlib import Path

DATE_VARIATIONS = [
    r"\d{1,2}(?:st|nd|rd|th)?\s*,?\s*(?:of\s+)?[A-Za-z]+,?\s*\d{4}",
    r"\d{1,2}\s*(?:st|nd|rd|th)?\s*(?:of\s+)?[A-Za-z]+,?\s*\d{4}",
    r"\d{1,2}(?:st|nd|rd|th)?\s*,?\s+[A-Za-z]+,?\s*\d{4}",
    r"\d{1,2}\.\d{1,2}\.\d{4}",
    r"\d{1,2}\s*[-./]\s*\d{1,2}(?:\s*[-./]\s*\d{2,4})?",
    r"\d{4}\s*[-./]\s*\d{1,2}\s*[-./]\s*\d{1,2}",
    r"\d{1,2}(?:st|nd|rd|th)?\s*[–\-]?\s*[A-Za-z]+\s*[–\-]?\s*\d{4}",
    r"((?:\d\s*){1,2}\s*\.\s*(?:\d\s*){1,2}\s*\.\s*(?:\d\s*){4})",
    r"\d{4}\s*[-./]\s*\d{1,2}\s*(?:[-./]\s*\d{0,2})?",
    r"\d{1,2}\s*[-./~]\s*\d{1,2}\s*[-./~]\s*\d{2,4}\.?",
    r"\d{4}\s*[–\-]\s*\d{1,2}\s*[–\-]\s*\d{1,2}",
    r"\d{1,2}\.\d{1,2}\.\d{4}",
    r"\b\d{4}\b",
    r"\d{1,2}\s*\.\s*\d{1,2}\s*\.\s*\d{4}\.?",
]

# === Metadata phrases ===
METADATA_PHRASES = [
    r"DECIDED\s*ON",
    r"Decided\s*on",
    r"Judgement\s*on",
    r"Decide\s*On",
    r"DECIDEN\s*ON",
    r"Decided\s*:?",
    r"DECIDEDON",
    r"ARGUED\s*&\s*(?:DECIDED)?(?:\s*ON)?",
    r"ARGUED\s*AND\s*(?:DECIDED)?(?:\s*ON)?",
    r"Judgment\s*(?:delivered\s*on|on)",
    r"Order\s*(?:Delivered\s*on|on)",
    r"Delivered\s*on",
    r"DATE\s*OF\s*JUDGMENT",
    r"Date",
    r"Judgment\s*(?:pronounced\s*on|delivered\s*on|on)",
    r"ARGUED\s*,?\s*DECIDED\s*(?:AND\s*JUDGMENT\s*PRONOUNCED\s*)?ON",
]


### Meta Split Function

In [2]:
# === Body start heuristics ===
def is_likely_body_start(text_after):
    """Heuristic check for where body begins."""
    snippet = text_after.strip()[:80]
    # Remove leading dots, stars, bullets
    cleaned = re.sub(r"^[\s\.\*\-–—_•.·~]+", "", snippet)
    cleaned = re.sub(r"\s{2,}", " ", cleaned)
    # Check if starts with capitalized words
    return bool(re.match(r"^[A-Z]", cleaned))


def candidate_strength(
    phrase, date_found, body_start_ok, pos, text_after=None, date_distance=0
):
    """
    Assign a logical score based on realistic hierarchy and context clues.
    - Considers phrase semantics, presence of date, body-start cues, and capitalization after date.
    - Penalizes large gaps between phrase/date and body start.
    """
    score = 0
    phrase_lower = phrase.lower()

    # === Base phrase weights ===
    # if "decided on" in phrase_lower:
    #     score += 50
    # elif "delivered" in phrase_lower:
    #     score += 38
    # elif "judgment" in phrase_lower or "judgement" in phrase_lower:
    #     score += 35
    # elif "order" in phrase_lower:
    #     score += 28
    # elif "argued" in phrase_lower:
    #     score += 20
    # else:
    #     score += 10

    # === Bonus for valid context ===
    if date_found:
        score += 30
    if body_start_ok:
        score += 25
    if not body_start_ok:
        score -= 25

    return score


def split_metadata_body(text):
    """
    Enhanced metadata/body splitter:
    - Handles 'phrase → date' and 'date → phrase' patterns
    - Fallback only for 'decided on'
    - Uses candidate strength scoring
    """
    candidates = []

    # === phrase → date ===
    for phrase in METADATA_PHRASES:
        # allow normal punctuation OR the special double-colon case
        pattern = rf"({phrase}\s*(?::\s*:|[:;\-–—])*\s*)"
        for m in re.finditer(pattern, text, re.IGNORECASE):
            after_phrase = text[m.end() :]
            for date_pat in DATE_VARIATIONS:
                date_m = re.match(rf"\s*{date_pat}", after_phrase)
                if date_m:
                    split_idx = m.end() + date_m.end()
                    tail = text[split_idx:]
                    body_ok = is_likely_body_start(tail)
                    candidates.append(
                        {
                            "phrase": phrase,
                            "date_found": True,
                            "body_ok": body_ok,
                            "score": candidate_strength(phrase, True, body_ok, m.end()),
                            "split_idx": split_idx,
                            "tail": tail[:20],
                        }
                    )
                    break

    # === date → phrase ===
    for date_pat in DATE_VARIATIONS:
        for m in re.finditer(rf"({date_pat}\s*[:;\-–—]*\s*)", text):
            after_date = text[m.end() :]
            for phrase in METADATA_PHRASES:
                # phrase_m = re.match(rf"\s*{phrase}", after_date, re.IGNORECASE)
                phrase_m = re.match(
                    rf"[\s\.\*\-–—_•·~]*{phrase}", after_date, re.IGNORECASE
                )
                if phrase_m:
                    split_idx = m.end() + phrase_m.end()
                    tail = text[split_idx:]
                    body_ok = is_likely_body_start(tail)
                    candidates.append(
                        {
                            "phrase": phrase,
                            "date_found": True,
                            "body_ok": body_ok,
                            "score": candidate_strength(phrase, True, body_ok, m.end()),
                            "split_idx": split_idx,
                            "tail": tail[:20],
                        }
                    )
                    break

    # fallback 'decided on' without date (underextracted document) ===
    if not candidates:
        for phrase in METADATA_PHRASES:
            if not re.search(r"\bDecided\s*On\b", phrase, re.IGNORECASE):
                continue
            for m in re.finditer(rf"({phrase}\s*[:;\-–—]*\s*)", text, re.IGNORECASE):
                split_idx = m.end()
                tail = text[split_idx:]
                body_ok = is_likely_body_start(tail)
                candidates.append(
                    {
                        "phrase": phrase,
                        "date_found": False,
                        "body_ok": body_ok,
                        "score": candidate_strength(phrase, False, body_ok, m.end()),
                        "split_idx": split_idx,
                        "tail": tail[:20],
                    }
                )

    if not candidates:
        return None, text, []

    # === Select best candidate ===
    if not candidates:
        return None, text, []

    # === Select best candidate with top-two-close-scores rule ===
    sorted_candidates = sorted(candidates, key=lambda c: c["score"], reverse=True)
    top1 = sorted_candidates[0]
    top2 = sorted_candidates[1] if len(sorted_candidates) > 1 else None

    if (
        top2
        and top1["score"] > 54
        and top2["score"] > 54
        and abs(top1["score"] - top2["score"]) <= 10
    ):
        # pick the one that comes first in the text
        best = min([top1, top2], key=lambda c: c["split_idx"])
    else:
        best = top1

    split_idx = best["split_idx"]
    return text[:split_idx].strip(), text[split_idx:].strip(), candidates



### Process Directory

In [3]:
def clean_body(text):
    """Clean the body text before writing."""
    text = text.strip()
    text = re.sub(r"^[\s\.\*\-–—_•.·~]+", "", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text


def process_dir(input_dir, output_dir, log_file="unmatched_files.log"):

    unmatched_files = []

    for path in input_dir.rglob("*.txt"):
        text = path.read_text(errors="ignore")
        meta, body, _ = split_metadata_body(text)

        if not meta:
            unmatched_files.append(str(path))

        body_clean = clean_body(body) if body else ""

        # Prepare output filename
        orig_name = path.stem
        if orig_name.endswith(".translated"):
            orig_name = orig_name.replace(".translated", "")
        out_filename = orig_name + ".metasplit.txt"

        # Keep subdir structure
        rel_path = path.relative_to(input_dir).parent
        out_path = output_dir / rel_path / out_filename
        out_path.parent.mkdir(parents=True, exist_ok=True)

        with out_path.open("w", encoding="utf-8") as f:
            if meta:
                f.write("=== METADATA ===\n")
                f.write(meta + "\n\n")
            f.write("=== BODY ===\n")
            f.write(body_clean + "\n")
        print(f"✅ Processed {os.path.basename(path)}")

    # Write unmatched log
    if unmatched_files:
        log_path = output_dir / log_file
        with log_path.open("w", encoding="utf-8") as logf:
            for file_path in unmatched_files:
                logf.write(file_path + "\n")
        print(f"Logged {len(unmatched_files)} unmatched files to {log_path}")

input_dir = Path("/kaggle/input/tamil-context")
output_dir = Path("/kaggle/working/Meta Split")
output_dir.mkdir(parents=True, exist_ok=True)

process_dir(input_dir, output_dir)

### Join Body into single block

In [4]:
import os
import re


class BodyLineJoiner:
    def __init__(self):
        pass

    def join_body_lines(self, text: str) -> str:
        # Split at === BODY ===
        parts = re.split(r"(=== BODY ===)", text)
        new_parts = []

        i = 0
        while i < len(parts):
            part = parts[i]
            if part == "=== BODY ===" and i + 1 < len(parts):
                new_parts.append(part)
                body_content = parts[i + 1]
                # Join all lines into a single line
                body_content = " ".join(
                    line.strip() for line in body_content.splitlines() if line.strip()
                )
                new_parts.append(body_content)
                i += 2
            else:
                new_parts.append(part)
                i += 1

        return "\n".join(new_parts)

    def process_file(self, input_file: str, output_file: str):
        with open(input_file, "r", encoding="utf-8") as f:
            content = f.read()
        processed = self.join_body_lines(content)
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(processed)
        print(f"Processed {input_file}")

    def process_directory(self, input_dir: str, output_dir: str):
        """Recursively process all .txt files in input_dir using self.process_file()."""
        input_dir = os.path.abspath(input_dir)
        output_dir = os.path.abspath(output_dir)
        os.makedirs(output_dir, exist_ok=True)
    
        for root, _, files in os.walk(input_dir):
            for filename in files:
                if not filename.lower().endswith(".txt"):
                    continue
    
                input_path = os.path.join(root, filename)
    
                # Preserve relative folder structure
                rel_path = os.path.relpath(input_path, input_dir)
                output_path = os.path.join(output_dir, rel_path)
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
                self.process_file(input_path, output_path)

        print("🎉 All files processed!")



INPUT_DIR = "/kaggle/working/Meta Split"
OUTPUT_DIR = "/kaggle/working/Body Joined"
joiner = BodyLineJoiner()
joiner.process_directory(INPUT_DIR, OUTPUT_DIR)


🎉 All files processed!


In [5]:
!zip -r "/kaggle/working/Body_Joined.zip" "/kaggle/working/Body Joined"

  adding: kaggle/working/Body Joined/ (stored 0%)
